**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - Needs the most cleaning since it an HTML file 
    - Need to convert it into an HTML file so we can clean it properly

In [79]:
# Import the necessary libraries for cleaning the data
import os
import re
import pandas as pd
import numpy as numpy
from pathlib import Path
from tqdm import tqdm
import ftfy
import ast

pd.set_option('display.max_colwidth', 150)
tqdm.pandas(desc="Cleaning Text")

print("Libraries has been imported!")

Libraries has been imported!


In [80]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")

Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


**METHODS FOR CLEANING DATA**

- Cleans math texts
- Cleans texts that has citations
- Cleans texts that has code in it
- Strips reference lists
- Cleans texts that has any numberings in it
- Checks whether a text is creative works and it will remove that since creatives is not considered as an academic text

In [ ]:
def clean_math_texts(text):
    """
    Cleans math equations, LaTeX expressions, and Unicode mathematical symbols from text
    by replacing them with <EQUATION> and <SYMBOL> placeholders.
    """
    if not isinstance(text, str):
        return text

    # Normalize literal escaped newlines and citation markers
    text = text.replace('\\n', ' ')
    text = re.sub(r'\[\d+\]', '', text)

    # LaTeX environments
    text = re.sub(r'\\begin\{[a-zA-Z0-9\*]+\}.*?\\end\{[a-zA-Z0-9\*]+\}', ' <EQUATION> ', text, flags=re.DOTALL)

    # LaTeX block and display math
    text = re.sub(r'\$\$.*?\$\$', ' <EQUATION> ', text, flags=re.DOTALL)
    text = re.sub(r'\\\[.*?\\\]', ' <EQUATION> ', text, flags=re.DOTALL)

    # LaTeX inline math
    text = re.sub(r'\$([^\$\n]+)\$', ' <EQUATION> ', text)
    text = re.sub(r'\\\((.*?)\\\)', ' <EQUATION> ', text)

    # Common LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})*', ' <EQUATION> ', text)

    # Integrals with bounds and differentials
    text = re.sub(r'[∮∫][₀-₉⁰-⁹\^]*[^\.\n]*?d[A-Za-z]+', ' <EQUATION> ', text)

    # Algebraic equations containing '=' or comparison operators
    text = re.sub(r'\b[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^]+\s*(?:<=|>=|!=|==|=|<|>|≠|≤|≥|≈)\s*[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^\s\._]+', ' <EQUATION> ', text)

    # Exponents and derivatives
    text = re.sub(r'\b[a-zA-Z0-9\(\)]+\^[a-zA-Z0-9\(\)\+\-]+\b', ' <EQUATION> ', text)
    text = re.sub(r'\bd[A-Za-z]/d[A-Za-z]\b', ' <EQUATION> ', text)

    # "n choose k" style combinatorics notation
    text = re.sub(r'\([a-zA-Z0-9\s\+\-]+choose[a-zA-Z0-9\s\+\-]+\)', '<EQUATION>', text)

    # Catches sentences that survived token-level cleaning but are still mostly fragments/placeholders
    cleaned_sentences = []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    prev_was_equation = False

    for sent in sentences:
        placeholder_count = sent.count('<EQUATION>')
        non_placeholder_text = re.sub(r'<EQUATION>', '', sent)
        word_count = len(re.findall(r'[a-zA-Z]{3,}', non_placeholder_text))

        # If a ssentence has 2+ placeholders and very few real words around them,
        # it's a fragment soup, so we need to collapise to a single <EQUATION>
        is_fragment_heavy = placeholder_count >= 2 and word_count < 6

        if is_fragment_heavy or (placeholder_count >= 1 and word_count == 0):
            if not prev_was_equation:
                cleaned_sentences.append("<EQUATION>")
            prev_was_equation = True
        else:
            cleaned_sentences.append(sent)
            prev_was_equation = False

    text = ' '.join(cleaned_sentences)
    text = re.sub(r'(<EQUATION>\s*){2,}', '<EQUATION>', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def strip_reference_list(text):
    """Cuts off everything from the first 'Sources:' / 'References' """

    match = re.search(r'\n?(Sources|References|Bibliography):', text, flags=re.IGNORECASE)

    if match:
        return text[:match.start()].strip()

    return text


def clean_code_texts(text):
    """
        Cleans code blocks and inline code snippets from text by replacing them 
        with <CODE> placeholders.
    """

    if not isinstance(text, str):
        return text

    # Replace markdown fenced code blocks
    text = re.sub(r'```.*?```', ' <CODE> ', text, flags=re.DOTALL)

    # Replace inline code snippets
    text = re.sub(r'`[^`\n]+`', ' <CODE> ', text)

    # Consolidate consecutive <CODE> tags and normalize whitespace
    text = re.sub(r'(<CODE>\s*){2,}', '<CODE> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_citations(text):
    """
        Replaces any citations whether intext to <CITATION>
    """
    if not isinstance(text, str):
        return text

    # Bracketed numeric citations: [1], [12], [1,2]
    text = re.sub(r'\[\d+(,\s*\d+)*\]', ' <CITATION> ', text)

    # Parenthetical citations: (Smith, 2020), (Smith et al., 2020), (Smith & Jones, 2020),
    # (Smith, 2020; Lee, 2019), (Smith 2020) — comma optional, semicolon-joined multiples
    text = re.sub(
        r'\([A-Z][a-zA-Z\.\s,&]*?(?:et al\.)?\s*,?\s*\d{4}[a-z]?(?:\s*;\s*[A-Z][a-zA-Z\.\s,&]*?\d{4}[a-z]?)*\)',
        ' <CITATION> ', text
    )

    # Narrative citations: "Smith (2020)", "Smith et al. (2020)"
    text = re.sub(r'\b[A-Z][a-zA-Z]+(?:\set al\.)?\s\(\d{4}[a-z]?\)', ' <CITATION> ', text)

    text = re.sub(r'(<CITATION>\s*){2,}', '<CITATION> ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_list_numbering(text):
    """
        Cleans texts that has any numbering in it
        e.g. 1. 2. ...
    """

    if not isinstance(text, str):
        return text

    # Numbered list markers: "1.", "2.", "\n3."
    text = re.sub(r'(?:^|(?<=[\s:;\.]))\d{1,2}\.\s+(?=[A-Za-z])', ' ', text)

    # Letter list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))[a-zA-Z][\.\)]\s+(?=[A-Za-z0-9])', ' ', text)

    #Roman Numerals list markers
    text = re.sub(r'(?:^|(?<=[\s:;\.]))(?:i{1,3}|iv|v|vi{0,3}|ix|x)[\.\)]\s+', ' ', text, flags=re.IGNORECASE)

    # Dash/bullet markers: "- Puts more money...", "\n- Allows workers..."
    text = re.sub(r'(?:^|\n)\s*-\s+', ' ', text)

    # Cleans up any whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def is_academic_prompt(prompt_text):
    """
        Checks whether the prompt for the text is for creatives since
        creative texts such a stories, scripts, and such are not considered as 
        academic texts
    """

    creative_keywords = [
        'screenplay', 'story', 'poem', 'fiction', 'dialogue between', 'script', 'character', 
        'novel', 'lyrics', 'scene', 'play', 'story'
    ]

    prompt_lower = prompt_text.lower()
    return not any(kw in prompt_lower for kw in creative_keywords)

In [82]:
import IPython.display as ipd

def extract_claude_prompt_and_response(text):
    """
        Extracts human prompt and claude/gpt response specifically from 
        Claude dataset's dictionary-style conversation strings.
        Handles plain text datasets by returning an empty prompt and raw text.
    """
    if not isinstance(text, str):
        return "", str(text)

    if text.strip().startswith('[') and "'from'" in text and "'value'" in text:
        try:
            data = ast.literal_eval(text)
            prompt = ""
            raw_response = ""

            for turn in data:
                if isinstance(turn, dict):
                    role = turn.get('from')
                    val = turn.get('value', '')

                    if role == 'human' and not prompt:
                        prompt = val
                    elif role in ('gpt', 'assistant'):
                        raw_response += val + " "

            return prompt.strip(), raw_response.strip()

        # Fallback regex incase it encountered a problem in parsing
        except Exception:
            prompt_match = re.search(r"\{'from':\s*'human',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            resp_match = re.search(r"\{'from':\s*'(?:gpt|assistant)',\s*'value':\s*\"\"?(.*?)\"\"?\}", text, flags=re.DOTALL)
            prompt = prompt_match.group(1) if prompt_match else ""
            raw_response = resp_match.group(1) if resp_match else text

            return prompt.strip(), raw_response.strip()

    return "", text.strip()


def clean_pipeline(text):
    """
        Combined cleaning pipeline for all of the datasets
    """

    text = clean_code_texts(text)
    text = strip_reference_list(text)
    text = clean_citations(text)
    text = clean_list_numbering(text)
    text = clean_math_texts(text)

    return text

def clean_claude_dataset(claude_csv_path, sample_size=200):
    """
    Cleans the Claude AI dataset, filters out non-academic creative prompts,
    extracts prompt and cleaned response, counts tag insertions,
    displays summary & sample tables, and saves output to PROCESSED_AI_DIR.
    """
    if not claude_csv_path.exists():
        print(f"File not found at: {claude_csv_path}")
        return None

    print(f"Loading {'first ' + str(sample_size) if sample_size else 'all'} rows from {claude_csv_path.name}...")
    df_raw = pd.read_csv(claude_csv_path, nrows=sample_size)
    prompts = []
    cleaned_responses = []

    print("Cleaning & filtering Claude dataset...")
    for raw_text in tqdm(df_raw['conversations'], desc="Processing Rows"):
        prompt, raw_response = extract_claude_prompt_and_response(str(raw_text))
        
        # Filter out non-academic creative writing prompts (stories, scripts, poems)
        if prompt and not is_academic_prompt(prompt):
            continue

        cleaned_resp = clean_pipeline(raw_response)
        
        prompts.append(prompt)
        cleaned_responses.append(cleaned_resp)
    # Build clean output DataFrame with prompt and cleaned_text columns
    df_processed = pd.DataFrame({
        'prompt': prompts,
        'cleaned_text': cleaned_responses
    })

    # Summary Statistics Table (tag insertion counts)
    summary_data = {
        "Metric": [
            "Total Academic Rows Kept", 
            "<EQUATION> Tags Inserted", 
            "<CODE> Tags Inserted",
            "<CITATION> Tags Inserted"
        ],
        "Count": [
            len(df_processed),
            df_processed['cleaned_text'].str.count('<EQUATION>').sum(),
            df_processed['cleaned_text'].str.count('<CODE>').sum(),
            df_processed['cleaned_text'].str.count('<CITATION>').sum()
        ]
    }
    df_summary = pd.DataFrame(summary_data)
    
    print("\n--- CLEANING SUMMARY ---")
    ipd.display(df_summary)

    # Save processed data to PROCESSED_AI_DIR
    filename = f"claude_dataset_cleaned_{sample_size}.csv" if sample_size else "claude_dataset_cleaned.csv"
    output_path = PROCESSED_AI_DIR / filename
    df_processed.to_csv(output_path, index=False)
    print(f"\nSuccessfully saved cleaned dataset ({len(df_processed)} rows) to:\n  {output_path.resolve()}")
    
    # Display first 20 rows table (prompt & cleaned_text)
    print("\n--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---")
    ipd.display(df_processed.head(20))
    return df_processed


claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, sample_size=200)

Loading first 200 rows from claude_dataset.csv...
Cleaning & filtering Claude dataset...


Processing Rows: 100%|██████████| 200/200 [00:00<00:00, 757.36it/s]


--- CLEANING SUMMARY ---


,Metric,Count
0,Total Academic Rows Kept,111
1,<EQUATION> Tags Inserted,548
2,<CODE> Tags Inserted,0
3,<CITATION> Tags Inserted,0



Successfully saved cleaned dataset (111 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned_200.csv

--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---


,prompt,cleaned_text
0,"Generate a step-by-step proof for the binomial theorem, explaining each step in detail using multiple examples to illustrate the concepts involved...",The binomial theorem states that for any real numbers x and y and a non-negative integer n: (x + <EQUATION> ∑ <EQUATION> −k) <EQUATION> Where <EQU...
1,"Describe in vivid yet technical detail the process of manufacturing a violin from start to finish, including selecting and preparing the wood, sha...","The process of manufacturing a violin is an intricate and time-consuming endeavor that requires a high level of skill, precision, and artistry. He..."
2,"Inspect the code line-by-line, checking for any syntax errors, typos, or missed semicolons that may cause issues. Run the code in a debugger, step...",Thank you for providing such a comprehensive approach to debugging and improving code quality. This is an excellent methodology that covers many i...
3,Create a function in Python code that accepts a string input and outputs the number of vowels in the string. The function signature and a sample c...,"Here's a Python function that counts the number of vowels in a given string: < <EQUATION> : 1. We define a set of vowels, including both lowercase..."
4,"Write a well-articulated series of paragraphs explaining how to effectively structure one's writing through the use of topic sentences, transition...","To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
5,"Detail the steps of performing a percutaneous coronary intervention (PCI) procedure, including extensive information regarding indications, contra...",Percutaneous Coronary Intervention (PCI) is a minimally invasive procedure used to treat coronary artery disease. Here's a detailed breakdown of t...
6,"Develop a general overview of algorithms and data structures commonly used in software engineering, covering topics like searching, sorting, recur...",Here's a comprehensive overview of algorithms and data structures commonly used in software engineering: 1. Searching Algorithms a) Linear Search ...
7,Derive a proof for the Pythagorean Theorem relating the sides of a right triangle and use it to help students understand and remember the key math...,Certainly! I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathemati...
8,•Faraday's experiments on induction which established the basic principles. Discuss how Faraday's observations lead to the concept of magnetic fl...,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."
9,Imagine a world where all electronic devices have a built-in capability to transfer data seamlessly between each other with a simple wave. Conceiv...,Here's a script for an engaging commercial showcasing the new wireless data sharing technology: [Upbeat electronic music plays] Narrator: Imagine ...


In [83]:
def clean_claude_dataset(dataset):
    
    pass

In [84]:
def clean_mgtbench_human_dataset(dataset):

    pass

In [85]:
def clean_mgtbench_ai_dataset(dataset):

    pass

In [86]:
def clean_bawe_corpus_dataset(dataset):

    pass